In [1]:
# Cell 1 — Imports
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
import random
import time
import os
import warnings
warnings.filterwarnings('ignore')

print("All imports successful")

All imports successful


In [2]:
# Cell 2 — Define the Bin Packing Problem
# The Grocery Bagger problem is a classic bin packing problem
# Items (groceries) must be packed into bags (bins) of fixed capacity
# Goal: minimize the number of bags used

# Simulate a grocery order
random.seed(42)
BAG_CAPACITY = 10  # kg — max weight per bag

# Generate random grocery items with weights
grocery_items = [
    {'name': 'Milk',        'weight': 1.89, 'fragile': False},
    {'name': 'Bread',       'weight': 0.68, 'fragile': True},
    {'name': 'Eggs',        'weight': 0.73, 'fragile': True},
    {'name': 'Chicken',     'weight': 1.36, 'fragile': False},
    {'name': 'Rice',        'weight': 2.27, 'fragile': False},
    {'name': 'Pasta',       'weight': 0.91, 'fragile': False},
    {'name': 'Tomatoes',    'weight': 0.45, 'fragile': True},
    {'name': 'Cheese',      'weight': 0.57, 'fragile': False},
    {'name': 'Butter',      'weight': 0.45, 'fragile': False},
    {'name': 'Juice',       'weight': 1.93, 'fragile': False},
    {'name': 'Chips',       'weight': 0.34, 'fragile': True},
    {'name': 'Detergent',   'weight': 1.81, 'fragile': False},
    {'name': 'Apples',      'weight': 0.91, 'fragile': False},
    {'name': 'Yogurt',      'weight': 0.68, 'fragile': False},
    {'name': 'Coffee',      'weight': 0.57, 'fragile': False},
    {'name': 'Cereal',      'weight': 0.45, 'fragile': True},
    {'name': 'Soap',        'weight': 0.23, 'fragile': False},
    {'name': 'Water',       'weight': 3.63, 'fragile': False},
    {'name': 'Frozen Pizza','weight': 0.91, 'fragile': False},
    {'name': 'Ice Cream',   'weight': 0.57, 'fragile': False},
]

items_df = pd.DataFrame(grocery_items)
print(f"Total items:        {len(items_df)}")
print(f"Total weight:       {items_df['weight'].sum():.2f} kg")
print(f"Bag capacity:       {BAG_CAPACITY} kg")
print(f"Fragile items:      {items_df['fragile'].sum()}")
print(f"Min bags needed:    {int(np.ceil(items_df['weight'].sum() / BAG_CAPACITY))}")
print(f"\nItem list:")
print(items_df.to_string(index=False))

Total items:        20
Total weight:       21.34 kg
Bag capacity:       10 kg
Fragile items:      5
Min bags needed:    3

Item list:
        name  weight  fragile
        Milk    1.89    False
       Bread    0.68     True
        Eggs    0.73     True
     Chicken    1.36    False
        Rice    2.27    False
       Pasta    0.91    False
    Tomatoes    0.45     True
      Cheese    0.57    False
      Butter    0.45    False
       Juice    1.93    False
       Chips    0.34     True
   Detergent    1.81    False
      Apples    0.91    False
      Yogurt    0.68    False
      Coffee    0.57    False
      Cereal    0.45     True
        Soap    0.23    False
       Water    3.63    False
Frozen Pizza    0.91    False
   Ice Cream    0.57    False


In [3]:
# Cell 3 — Three Packing Algorithms
def first_fit(items, capacity):
    """Greedy: place each item in first bag it fits."""
    bags = []
    for item in items:
        placed = False
        for bag in bags:
            if sum(i['weight'] for i in bag) + item['weight'] <= capacity:
                bag.append(item)
                placed = True
                break
        if not placed:
            bags.append([item])
    return bags

def first_fit_decreasing(items, capacity):
    """Sort by weight descending then apply First Fit — better packing."""
    sorted_items = sorted(items, key=lambda x: x['weight'], reverse=True)
    return first_fit(sorted_items, capacity)

def fragile_first(items, capacity):
    """
    Custom algorithm: pack fragile items first in their own bags,
    then fill remaining space with non-fragile items.
    Mirrors the Java grocery bagger logic.
    """
    fragile     = [i for i in items if i['fragile']]
    non_fragile = [i for i in items if not i['fragile']]
    
    # Sort both by weight descending
    fragile     = sorted(fragile,     key=lambda x: x['weight'], reverse=True)
    non_fragile = sorted(non_fragile, key=lambda x: x['weight'], reverse=True)
    
    bags = []
    # Place fragile items first
    for item in fragile:
        placed = False
        for bag in bags:
            bag_weight   = sum(i['weight'] for i in bag)
            has_fragile  = any(i['fragile'] for i in bag)
            if bag_weight + item['weight'] <= capacity:
                bag.append(item)
                placed = True
                break
        if not placed:
            bags.append([item])
    
    # Fill remaining space with non-fragile
    for item in non_fragile:
        placed = False
        for bag in bags:
            if sum(i['weight'] for i in bag) + item['weight'] <= capacity:
                bag.append(item)
                placed = True
                break
        if not placed:
            bags.append([item])
    return bags

# Run all three algorithms
items = grocery_items.copy()

ff_bags  = first_fit(items, BAG_CAPACITY)
ffd_bags = first_fit_decreasing(items, BAG_CAPACITY)
frf_bags = fragile_first(items, BAG_CAPACITY)

def summarize(bags, name):
    print(f"\n=== {name} ===")
    print(f"Bags used: {len(bags)}")
    for i, bag in enumerate(bags):
        weight = sum(item['weight'] for item in bag)
        names  = [item['name'] for item in bag]
        frags  = [item['name'] for item in bag if item['fragile']]
        print(f"  Bag {i+1}: {weight:.2f}kg — {names}")
        if frags:
            print(f"          Fragile: {frags}")

summarize(ff_bags,  "First Fit (Greedy)")
summarize(ffd_bags, "First Fit Decreasing")
summarize(frf_bags, "Fragile First (Java Logic)")


=== First Fit (Greedy) ===
Bags used: 3
  Bag 1: 9.88kg — ['Milk', 'Bread', 'Eggs', 'Chicken', 'Rice', 'Pasta', 'Tomatoes', 'Cheese', 'Butter', 'Chips', 'Soap']
          Fragile: ['Bread', 'Eggs', 'Tomatoes', 'Chips']
  Bag 2: 9.98kg — ['Juice', 'Detergent', 'Apples', 'Yogurt', 'Coffee', 'Cereal', 'Water']
          Fragile: ['Cereal']
  Bag 3: 1.48kg — ['Frozen Pizza', 'Ice Cream']

=== First Fit Decreasing ===
Bags used: 3
  Bag 1: 9.95kg — ['Water', 'Rice', 'Juice', 'Milk', 'Soap']
  Bag 2: 9.70kg — ['Detergent', 'Chicken', 'Pasta', 'Apples', 'Frozen Pizza', 'Eggs', 'Bread', 'Yogurt', 'Cheese', 'Coffee', 'Ice Cream']
          Fragile: ['Eggs', 'Bread']
  Bag 3: 1.69kg — ['Tomatoes', 'Butter', 'Cereal', 'Chips']
          Fragile: ['Tomatoes', 'Cereal', 'Chips']

=== Fragile First (Java Logic) ===
Bags used: 3
  Bag 1: 9.91kg — ['Eggs', 'Bread', 'Tomatoes', 'Cereal', 'Chips', 'Water', 'Rice', 'Chicken']
          Fragile: ['Eggs', 'Bread', 'Tomatoes', 'Cereal', 'Chips']
  Bag 2: 9

In [4]:
# Cell 4 — Efficiency Analysis & Visualization
def packing_efficiency(bags, capacity):
    total_weight = sum(sum(i['weight'] for i in bag) for bag in bags)
    total_space  = len(bags) * capacity
    return total_weight / total_space

algorithms = {
    'First Fit':           ff_bags,
    'First Fit Decreasing': ffd_bags,
    'Fragile First':       frf_bags
}

metrics = []
for name, bags in algorithms.items():
    efficiency = packing_efficiency(bags, BAG_CAPACITY)
    bag_weights = [sum(i['weight'] for i in bag) for bag in bags]
    metrics.append({
        'algorithm':  name,
        'bags_used':  len(bags),
        'efficiency': round(efficiency, 4),
        'avg_bag_weight': round(np.mean(bag_weights), 2),
        'min_bag_weight': round(min(bag_weights), 2),
        'max_bag_weight': round(max(bag_weights), 2)
    })

metrics_df = pd.DataFrame(metrics)
print("=== Algorithm Comparison ===")
print(metrics_df.to_string(index=False))

# Plot bags used
fig1 = px.bar(metrics_df, x='algorithm', y='bags_used',
              color='efficiency',
              color_continuous_scale='Viridis',
              title='Bags Used by Algorithm',
              template='plotly_dark')
fig1.update_layout(width=700, height=400)
fig1.show()

# Plot packing efficiency
fig2 = px.bar(metrics_df, x='algorithm', y='efficiency',
              color='efficiency',
              color_continuous_scale='Blues',
              title='Packing Efficiency by Algorithm (higher = better)',
              template='plotly_dark')
fig2.add_hline(y=1.0, line=dict(color='lime', dash='dash'),
               annotation_text='Perfect packing')
fig2.update_layout(width=700, height=400)
fig2.show()

# Export
output_dir = r'C:\Users\Gurveer\ds-portfolio\project-21-grocery-bagger\outputs'
os.makedirs(output_dir, exist_ok=True)
metrics_df.to_csv(f'{output_dir}\\algorithm_comparison.csv', index=False)
items_df.to_csv(f'{output_dir}\\grocery_items.csv', index=False)
print(f"\nExports saved to outputs/")

=== Algorithm Comparison ===
           algorithm  bags_used  efficiency  avg_bag_weight  min_bag_weight  max_bag_weight
           First Fit          3      0.7113            7.11            1.48            9.98
First Fit Decreasing          3      0.7113            7.11            1.69            9.95
       Fragile First          3      0.7113            7.11            1.59            9.91



Exports saved to outputs/


In [5]:
# Cell 5 — Summary
print("""
╔══════════════════════════════════════════════════════════╗
║         Grocery Bagger — Project Summary                 ║
╠══════════════════════════════════════════════════════════╣
║  Problem:     Bin Packing (NP-Hard optimization)         ║
║  Items:       20 groceries, 21.34 kg total               ║
║  Bag capacity: 10 kg                                     ║
║  Optimal bags: 3 (mathematically proven minimum)         ║
╠══════════════════════════════════════════════════════════╣
║  Algorithm Results:                                      ║
║  • First Fit:           3 bags, 71.1% efficiency         ║
║  • First Fit Decreasing: 3 bags, 71.1% efficiency        ║
║  • Fragile First:       3 bags, 71.1% efficiency         ║
╠══════════════════════════════════════════════════════════╣
║  Key insight: All algorithms reach optimal solution      ║
║  Fragile First adds safety constraint (Java logic)       ║
║  Origin: Java COMP 2000 group project (Eclipse)          ║
╚══════════════════════════════════════════════════════════╝
""")


╔══════════════════════════════════════════════════════════╗
║         Grocery Bagger — Project Summary                 ║
╠══════════════════════════════════════════════════════════╣
║  Problem:     Bin Packing (NP-Hard optimization)         ║
║  Items:       20 groceries, 21.34 kg total               ║
║  Bag capacity: 10 kg                                     ║
║  Optimal bags: 3 (mathematically proven minimum)         ║
╠══════════════════════════════════════════════════════════╣
║  Algorithm Results:                                      ║
║  • First Fit:           3 bags, 71.1% efficiency         ║
║  • First Fit Decreasing: 3 bags, 71.1% efficiency        ║
║  • Fragile First:       3 bags, 71.1% efficiency         ║
╠══════════════════════════════════════════════════════════╣
║  Key insight: All algorithms reach optimal solution      ║
║  Fragile First adds safety constraint (Java logic)       ║
║  Origin: Java COMP 2000 group project (Eclipse)          ║
╚══════════════════════